# W08 — Assignment (SQL limpieza + Many-to-Many)

**Único entregable semanal.**

## Setup

In [ ]:
from pathlib import Path
import duckdb

PROJECT_ROOT = Path(".").resolve()
RAW_CSV = PROJECT_ROOT / "data" / "raw" / "pscomppars.csv"
DB_PATH = PROJECT_ROOT / "data" / "exoplanets.duckdb"

if not RAW_CSV.exists():
    raise FileNotFoundError(f"Missing {RAW_CSV}. Run W01/W02 download first.")

def sql_path(p: Path) -> str:
    return "'" + p.resolve().as_posix().replace("'", "''") + "'"

con = duckdb.connect(str(DB_PATH))

con.execute("DROP VIEW IF EXISTS raw_ps")
con.execute(f"CREATE VIEW raw_ps AS SELECT * FROM read_csv_auto({sql_path(RAW_CSV)})")

con.sql("SELECT COUNT(*) AS n_raw FROM raw_ps").show()

## Parte A — Limpieza Raw→Silver v2

In [ ]:
con.sql("SELECT discoverymethod, COUNT(*) AS n FROM raw_ps WHERE discoverymethod IS NOT NULL GROUP BY discoverymethod ORDER BY n DESC LIMIT 15").show()
con.sql("SELECT COUNT(DISTINCT discoverymethod) AS n_unique_methods FROM raw_ps WHERE discoverymethod IS NOT NULL").show()

### TODO A1 — method_map

In [ ]:
# TODO A1: crea una tabla method_map con mapeos raw -> canonical (>=6).
con.execute("DROP TABLE IF EXISTS method_map")

con.execute("CREATE TABLE method_map(raw_method VARCHAR, canonical_method VARCHAR)")
con.execute("""INSERT INTO method_map VALUES
  ('Transit', 'transit'),
  ('transit', 'transit'),
  (' TRANSIT ', 'transit'),
  ('Radial Velocity', 'radial_velocity'),
  ('radial velocity', 'radial_velocity'),
  ('Imaging', 'imaging'),
  ('Microlensing', 'microlensing'),
  ('Timing', 'timing')""")

con.sql("SELECT * FROM method_map").show()


### TODO A2 — silver_planet_v2

In [ ]:
# TODO A2: crea silver_planet_v2 con:
# - hostname_clean = LOWER(TRIM(hostname))
# - discoverymethod_clean = COALESCE(map, LOWER(TRIM(discoverymethod_norm)))
# - disc_era = CASE por década
con.execute("DROP TABLE IF EXISTS silver_planet_v2")

con.execute("""CREATE TABLE silver_planet_v2 AS
WITH base AS (
  SELECT
    pl_name, hostname, discoverymethod, disc_year, sy_snum, sy_pnum, sy_dist, ra, dec,
    pl_orbper, pl_rade, pl_bmasse, pl_eqt, st_teff, st_rad, st_mass,
    LOWER(TRIM(hostname)) AS hostname_clean,
    NULLIF(TRIM(discoverymethod), '') AS discoverymethod_norm
  FROM raw_ps
  WHERE pl_name IS NOT NULL
    AND hostname IS NOT NULL
    AND (disc_year IS NULL OR (disc_year BETWEEN 1980 AND 2026))
    AND (pl_rade  IS NULL OR (pl_rade  > 0 AND pl_rade  <= 30))
    AND (pl_bmasse IS NULL OR (pl_bmasse > 0))
),
mapped AS (
  SELECT
    b.*,
    COALESCE(m.canonical_method, LOWER(TRIM(b.discoverymethod_norm))) AS discoverymethod_clean,
    CASE
      WHEN b.disc_year IS NULL THEN NULL
      WHEN b.disc_year < 1990 THEN 'pre_1990'
      WHEN b.disc_year < 2000 THEN '1990s'
      WHEN b.disc_year < 2010 THEN '2000s'
      WHEN b.disc_year < 2020 THEN '2010s'
      ELSE '2020s'
    END AS disc_era
  FROM base b
  LEFT JOIN method_map m
    ON m.raw_method = b.discoverymethod
)
SELECT
  pl_name, hostname, discoverymethod, disc_year, sy_snum, sy_pnum, sy_dist, ra, dec,
  pl_orbper, pl_rade, pl_bmasse, pl_eqt, st_teff, st_rad, st_mass,
  hostname_clean, discoverymethod_clean, disc_era
FROM mapped""")

con.sql("SELECT COUNT(*) AS n_rows FROM silver_planet_v2").show()
con.sql("SELECT COUNT(*) AS n_null_hosts FROM silver_planet_v2 WHERE hostname_clean IS NULL").show()
con.sql("SELECT discoverymethod_clean, COUNT(*) AS n FROM silver_planet_v2 WHERE discoverymethod_clean IS NOT NULL GROUP BY discoverymethod_clean ORDER BY n DESC LIMIT 15").show()


## Parte B — Many-to-Many (toy schema)

In [ ]:
# TODO B1: construye un ejemplo M:N (toy schema) y responde 2 preguntas.
#
# REQUISITO (para que cuente como completo):
# - Debes crear el esquema con **link table** + **PK/FK**:
#   - planet_demo(planet_id PRIMARY KEY, name NOT NULL)
#   - method_demo(method_id PRIMARY KEY, method_name UNIQUE NOT NULL)
#   - planet_method_demo(planet_id, method_id) con:
#       PRIMARY KEY (planet_id, method_id)
#       FOREIGN KEY (planet_id) REFERENCES planet_demo(planet_id)
#       FOREIGN KEY (method_id) REFERENCES method_demo(method_id)
#
# - Inserta al menos 4 planetas, 3 métodos y relaciones M:N (un planeta con 2 métodos).
#
# Nota idempotencia (FK-safe): dropea puente primero.
con.execute("DROP TABLE IF EXISTS planet_method_demo")
con.execute("DROP TABLE IF EXISTS method_demo")
con.execute("DROP TABLE IF EXISTS planet_demo")

con.execute("""
CREATE TABLE planet_demo(
  planet_id INTEGER PRIMARY KEY,
  name VARCHAR NOT NULL
);
""")
con.execute("""
CREATE TABLE method_demo(
  method_id INTEGER PRIMARY KEY,
  method_name VARCHAR NOT NULL UNIQUE
);
""")
con.execute("""
CREATE TABLE planet_method_demo(
  planet_id INTEGER NOT NULL,
  method_id INTEGER NOT NULL,
  PRIMARY KEY (planet_id, method_id),
  FOREIGN KEY (planet_id) REFERENCES planet_demo(planet_id),
  FOREIGN KEY (method_id) REFERENCES method_demo(method_id)
);
""")

con.execute("""
INSERT INTO planet_demo VALUES
  (1, 'Aurelia'),
  (2, 'Borealis'),
  (3, 'Cetus'),
  (4, 'Draco');
""")
con.execute("""
INSERT INTO method_demo VALUES
  (10, 'transit'),
  (20, 'radial_velocity'),
  (30, 'imaging');
""")
con.execute("""
INSERT INTO planet_method_demo VALUES
  (1, 10),
  (1, 20),
  (2, 10),
  (3, 30),
  (4, 20),
  (4, 30);
""")

# Q1: # planetas por método (usa COUNT(DISTINCT planet_id))
q1 = """
SELECT
  m.method_name,
  COUNT(DISTINCT pm.planet_id) AS n_planets
FROM method_demo m
JOIN planet_method_demo pm
  ON pm.method_id = m.method_id
GROUP BY m.method_name
ORDER BY n_planets DESC, m.method_name;
"""
con.sql(q1).show()

# Q2: # métodos por planeta
q2 = """
SELECT
  p.name,
  COUNT(DISTINCT pm.method_id) AS n_methods
FROM planet_demo p
JOIN planet_method_demo pm
  ON pm.planet_id = p.planet_id
GROUP BY p.name
ORDER BY n_methods DESC, p.name;
"""
con.sql(q2).show()


In [ ]:
# TODO B2 (REQUERIDO): check de duplicados en la link table (debe dar 0 filas)
# Si tu PK compuesta está bien, este check debería retornar vacío.

con.sql("""
SELECT planet_id, method_id, COUNT(*) AS c
FROM planet_method_demo
GROUP BY planet_id, method_id
HAVING COUNT(*) > 1
""").show()

# (Opcional) intenta insertar un duplicado para ver que la PK compuesta lo bloquea
# try:
#     con.execute("INSERT INTO planet_method_demo VALUES (1, 10)")
# except Exception as e:
#     print("OK (PK compuesta bloquea duplicado):", str(e).splitlines()[0])

## Entregable único semanal (W08)
- Ejecuta el assignment.
- Entrega:
  1) `docs/w08_report.md` (copiar template)
  2) 1 entrada nueva en `docs/decisions_log.md` (copiar template)

**Extra requerido (M:N):** incluye evidencia de PK/FK en tu DDL y/o el check `HAVING COUNT(*)>1` retornando vacío.